### 1. Import thư viện 

In [25]:
import pandas as pd
import numpy as np
import joblib
from tqdm.auto import tqdm

# Sentence Transformers để tạo embeddings
from sentence_transformers import SentenceTransformer

# Scikit-learn cho mô hình và đánh giá
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, make_scorer


### 2. Tải và khám phá dữ liệu 

In [26]:
try:
    df = pd.read_csv('data/mail_data.csv')
    print("Tải dữ liệu thành công!")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy tệp 'data/raw_data.csv'.")
    print("Hãy chắc chắn rằng bạn đã tạo thư mục 'data' và đặt tệp dữ liệu vào đó.")

df.head()


Tải dữ liệu thành công!


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6574 entries, 0 to 6573
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  6574 non-null   object
 1   Message   6574 non-null   object
dtypes: object(2)
memory usage: 102.8+ KB


In [28]:
df['Category'].value_counts()


Category
ham     5408
spam    1166
Name: count, dtype: int64

### 3. Tiền xử lý dữ liệu 

In [29]:
# Ánh xạ 'ham' -> 0 và 'spam' -> 1
df['label'] = df['Category'].map({'ham': 0, 'spam': 1})

# Xóa các dòng có giá trị thiếu (nếu có)
df.dropna(inplace=True)

# Chọn các cột cần thiết
data = df[['Message', 'label']]

print("Dữ liệu sau khi xử lý:")
data.head()


Dữ liệu sau khi xử lý:


,Message,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


### 4. Tạo đặc trưng với SENTENCE TRANSFORMERS

In [30]:
# Sử dụng mô hình `all-MiniLM-L6-v2` đã được huấn luyện trước 
model_name = 'all-MiniLM-L6-v2'
embedding_model = SentenceTransformer(model_name)

In [ ]:
# Tạo embeddings cho tất cả các tin nhắn
print("Bắt đầu tạo embeddings cho dữ liệu...")
corpus = data['Message'].tolist()

# Sử dụng tqdm để theo dõi tiến trình
embeddings = embedding_model.encode(corpus, show_progress_bar=True)
# Tự động chia dữ liệu thành batch nhỏ 

print(f"Đã tạo xong embeddings. Kích thước của mảng embeddings: {embeddings.shape}")


Bắt đầu tạo embeddings cho dữ liệu...


Batches:   0%|          | 0/206 [00:00<?, ?it/s]

Đã tạo xong embeddings. Kích thước của mảng embeddings: (6574, 384)


In [32]:
# Gán X (features) và y (target)
X = embeddings
y = data['label'].values

### 5. Chia dữ liệu, huấn luyện và tối ưu hóa mô hình 

In [ ]:
# Chia dữ liệu thành tập train và test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# starify đảm bảo tỷ lệ phân phối nhãn trong tập train và test giống nhau

print(f"train set: {X_train.shape[0]} samples")
print(f"test set: {X_test.shape[0]} samples")

train set: 5259 samples
test set: 1315 samples


In [ ]:
# Thiết lập mô hình và không gian siêu tham số để tìm kiếm
model_log_reg = LogisticRegression(penalty='l2', C=1.0, max_iter=10000, class_weight='balanced')
# class_weight='balanced' các mẫu thuốc lớp thiểu số thì có trong số cao hơn => phạt nhiều hơn trong quá trình huấn luyện

# 'C' là nghịch đảo của cường độ điều chuẩn. C cà ng nhỏ thì W Mô hình sẽ ưu tiên giữ cho các trọng số w nhỏ
linSpace = np.logspace(0, 100, 1000) 
param_grid = {'C': linSpace}

#Tạo một scorer chuyên dụng để tối ưu recall cho lớp spam (label=1)
spam_recall_scorer = make_scorer(recall_score, pos_label=1)


# Thiết lập GridSearchCV với 5-fold cross-validation
# Dữ liệu huấn luyện sẽ được chia thành 5 phần, 4 phần dùng để train và 1 phần để validation
grid_search = GridSearchCV(model_log_reg, param_grid, cv=5, scoring=spam_recall_scorer, verbose=1, n_jobs=-1)
# verbose = 1 để in ra thông tin quá trình tìm kiếm
# n_jobs = -1 để sử dụng tất cả các luồng CPU có sẵn

In [37]:
# Bắt đầu quá trình tìm kiếm và huấn luyện
print("Bắt đầu tìm kiếm siêu tham số tốt nhất với GridSearchCV...")
grid_search.fit(X_train, y_train)

Bắt đầu tìm kiếm siêu tham số tốt nhất với GridSearchCV...
Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


GridSearchCV(cv=5,
             estimator=LogisticRegression(class_weight='balanced',
                                          max_iter=1000),
             n_jobs=-1,
             param_grid={'C': array([1.00000000e+000, 1.25921561e+000, 1.58562396e+000, 1.99664245e+000,
       2.51420335e+000, 3.16592411e+000, 3.98658107e+000, 5.01996513e+000,
       6.32121848e+000, 7.95977700e+000, 1.00230755e+001, 1.26212131e+001,
       1.58928287e+001, 2.00124980e+001, 2.52000...
       1.25342427e+098, 1.57833141e+098, 1.98745955e+098, 2.50264010e+098,
       3.15136348e+098, 3.96824610e+098, 4.99687745e+098, 6.29214611e+098,
       7.92316862e+098, 9.97697764e+098, 1.25631660e+099, 1.58197348e+099,
       1.99204571e+099, 2.50841506e+099, 3.15863541e+099, 3.97740302e+099,
       5.00840799e+099, 6.30666554e+099, 7.94145172e+099, 1.00000000e+100])},
             scoring=make_scorer(recall_score, response_method='predict', pos_label=1),
             verbose=1)

In [39]:
print(f"Siêu tham số tốt nhất được tìm thấy: {grid_search.best_params_}")
print(f"Độ chính xác tốt nhất trên tập validation (cross-validated): {grid_search.best_score_:.4f}")

# Lấy ra mô hình tốt nhất
best_model = grid_search.best_estimator_

Siêu tham số tốt nhất được tìm thấy: {'C': np.float64(2.5142033481427974)}
Độ chính xác tốt nhất trên tập validation (cross-validated): 0.9753


### 6. Đánh giá mô hình trên tập kiểm tra 

In [47]:
# Dự đoán trên tập test
y_pred = best_model.predict(X_test)

# Tính toán các chỉ số đánh giá
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, target_names=['Ham', 'Spam'])

print(f"Độ chính xác trên tập kiểm tra: {accuracy:.4f}\n")
print("Confusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(class_report)

Độ chính xác trên tập kiểm tra: 0.9764

Confusion Matrix:
[[1059   23]
 [   8  225]]

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99      1082
        Spam       0.91      0.97      0.94       233

    accuracy                           0.98      1315
   macro avg       0.95      0.97      0.96      1315
weighted avg       0.98      0.98      0.98      1315



### 7. Lưu mô hình tốt nhất 

In [41]:
# Đường dẫn để lưu mô hình
model_path = 'models/logistic_regression_best.pkl'

# Sử dụng joblib để lưu mô hình
joblib.dump(best_model, model_path)

print(f"Mô hình đã được lưu tại: {model_path}")

Mô hình đã được lưu tại: models/logistic_regression_best.pkl


### 8. Sử dụng mô hình đã lưu 

In [42]:
# Tải lại mô hình (giả sử trong một phiên làm việc khác)
loaded_model = joblib.load(model_path)
# Tạo một vài email mới để kiểm tra
new_emails = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/claim-yours to claim now.", # Spam
    "Hi mom, I'll be home late for dinner tonight. Don't wait up for me.", # Ham
    "URGENT: Your account has been suspended. Please verify your details immediately to avoid closure." # Spam
]

# Tạo embeddings cho email mới
new_embeddings = embedding_model.encode(new_emails, show_progress_bar=False)

# Dự đoán
predictions = loaded_model.predict(new_embeddings)

# In kết quả
for email, pred in zip(new_emails, predictions):
    label = "Spam" if pred == 1 else "Ham"
    print(f"Email: \"{email[:50]}...\"\n  -> Dự đoán: {label}\n")

Email: "Congratulations! You've won a $1000 Walmart gift c..."
  -> Dự đoán: Spam

Email: "Hi mom, I'll be home late for dinner tonight. Don'..."
  -> Dự đoán: Ham

Email: "URGENT: Your account has been suspended. Please ve..."
  -> Dự đoán: Ham

